## Points to implement

* [x] Request and display website page
* [x] Automatically search tags from the text
* [x] Save class names for a given url
* [] Scrape titles and content (separately if possible)
* [~] Categorize titles by finding common keywords


In [3]:
import requests

url_str = 'https://www.lemonde.fr/'

r = requests.get(url_str, auth=('user', 'pass'))
r.status_code

200

In [21]:
html_code = r.content.decode('utf')
html_code

'                    <!DOCTYPE html> <html lang="fr" prefix="og: https://ogp.me/ns#"> <head> <meta charset="UTF-8"> <meta http-equiv="X-UA-Compatible" content="IE=edge"> <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no"> <meta name="referrer" content="no-referrer-when-downgrade"> <meta name="theme-color" content="#ffffff">  <script type="text/plain" data-gdpr-purposes="personalization" data-gdpr-src="https://www.lemonde.fr/bucket/resources/front/js/chartbeatMab.88d00fd6428d5de2.js" async="1"></script>    <link data-rh="true" rel="alternate" href="https://www.lemonde.fr/en/" hreflang="en-US"> <link data-rh="true" rel="alternate" href="https://www.lemonde.fr/en/" hreflang="en"> <link data-rh="true" rel="alternate" href="https://www.lemonde.fr/en/" hreflang="en-CA"> <link data-rh="true" rel="alternate" href="https://www.lemonde.fr/en/" hreflang="en-GB">  <link rel="preconnect" href="//img.lemde.fr">    <link rel="preload" as="image" fetchpriority="high"

In [163]:
from tempfile import NamedTemporaryFile
import webbrowser

def view_in_browser(html):
    with NamedTemporaryFile("wb", delete=False, suffix=".html") as file:
        file.write(html.encode('utf'))
    webbrowser.open_new_tab(f"file://{file.name}")

In [164]:
view_in_browser(html_code)

### Search for titles

In [22]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_code, 'html.parser')

In [ ]:
def custom_selector(tag, text_sample):
	return tag.name=='a' and text_sample in tag.text and ( tag.has_attr("class") or tag.has_attr("id") )


sample_text = "Dans "
containing_text = soup.find_all(lambda tag: custom_selector(tag, sample_text))

print(len(containing_text))
for t in containing_text:
    print(t)

1
<a aria-describedby="zone1-live-3459939" class="lmd-link-clickarea__link" href="https://www.lemonde.fr/societe/live/2026/02/19/en-direct-mort-de-quentin-deranque-il-reste-plusieurs-personnes-a-identifier-selon-le-procureur-de-lyon_6667365_3224.html"> <p class="article__title-label">Dans l’enquête sur la mort de Quentin Deranque, trois suspects mis en examen, dont l’assistant du député « insoumis »Raphaël Arnault</p> </a>


In [60]:
link_tag = containing_text[0]
link_tag.get_text()

' Dans l’enquête sur la mort de Quentin Deranque, trois suspects mis en examen, dont l’assistant du député «\xa0insoumis\xa0»Raphaël Arnault '

In [119]:
def count_children(main_tag):
    return len(main_tag.find_all())

def find_childmost_tags(main_tag):
    child_list = []
    for tag in main_tag.find_all():
        if count_children(tag) == 0:
            child_list.append(tag)
    return child_list

def find_childmost_with_text(main_tag):
    child_list = find_childmost_tags(main_tag)
    tags_with_text = []
    for tag in child_list:
        if tag.text != '':
            tags_with_text.append(tag)
    return tags_with_text

In [127]:
tags_with_text = find_childmost_with_text(soup)
len(tags_with_text)

762

In [130]:
sample_text = "Dans "
i = 0
for tag in tags_with_text:
    if sample_text in tag.text:
        selected_tag = tag
        i += 1
print(i)

1


In [146]:
def first_parent_with_class(main_tag):
    temp_tag = main_tag
    while temp_tag.has_attr('class') == False:
        temp_tag = temp_tag.parent
    return temp_tag

In [147]:
first_parent_with_class(selected_tag)

<p class="article__title-label">Dans l’enquête sur la mort de Quentin Deranque, trois suspects mis en examen, dont l’assistant du député « insoumis »Raphaël Arnault</p>

In [155]:
def get_class_with_string(tags_with_text_list, sample_str):
    i = 0
    for tag in tags_with_text_list:
        if sample_str in tag.text:
            selected_tag = tag
            i += 1
    if i > 1:
        print("Too many matches with the given string!")
        return 0
    tag_with_class = first_parent_with_class(selected_tag)
    return tag_with_class.attrs['class'][0]

In [158]:
class_string = get_class_with_string(tags_with_text, "Dans ")
print(class_string)

article__title-label


In [187]:
import copy

new_soup = copy.deepcopy(soup)

In [ ]:
strike_tag = new_soup.new_tag('font', attrs={'color':'lightgreen'})
for tag in new_soup.find_all(class_=class_string):
    tag.wrap(strike_tag)
print(new_soup.find(class_=class_string).parent)

<font color="lightgreen"><p class="article__title-label">Dans l’enquête sur la mort de Quentin Deranque, trois suspects mis en examen, dont l’assistant du député « insoumis »Raphaël Arnault</p></font>


In [190]:
view_in_browser(str(new_soup))

### Use pre-made methods to scrape all titles

In [69]:
import scrape_news_titles as scrape
from importlib import reload
reload(scrape_news_titles)

<module 'scrape_news_titles' from 'c:\\Users\\cmsup\\Documents\\Python\\news_scraping\\scrape_news_titles.py'>

In [2]:
import requests
from bs4 import BeautifulSoup

url_str = 'https://www.lemonde.fr/'

r = requests.get(url_str, auth=('user', 'pass'))
r.status_code

html_code = r.content.decode('utf')
soup = BeautifulSoup(html_code, 'html.parser')

In [30]:
scrape.view_in_browser(html_code)

In [ ]:
tags_with_text = scrape.find_childmost_with_text(soup)

In [48]:
string_list = [
    "La décision de la Cour",
    "Jeffrey Epstein possédait",
]

class_list = []
class_list = scrape.get_classes_with_strings(tags_with_text, string_list)
new_soup = scrape.highlight_titles(soup, class_list)

In [49]:
scrape.view_in_browser(new_soup)

In [50]:
print(class_list)

['article__title-label', 'article__title']


In [51]:
import requests
from bs4 import BeautifulSoup

url_str = 'https://www.lecanardenchaine.fr/'

r = requests.get(url_str, auth=('user', 'pass'))
r.status_code

html_code = r.content.decode('utf')
soup = BeautifulSoup(html_code, 'html.parser')

In [52]:
scrape.view_in_browser(html_code)

In [53]:
tags_with_text = scrape.find_childmost_with_text(soup)

In [54]:
string_list = [
    "Vincent Bolloré joue les",
]

class_list = []
class_list = scrape.get_classes_with_strings(tags_with_text, string_list)
new_soup = scrape.highlight_titles(soup, class_list)

In [55]:
scrape.view_in_browser(new_soup)

In [56]:
print(class_list)

['article-item__title']


In [105]:
import scrape_news_titles as scrape
from importlib import reload
reload(scrape_news_titles)

<module 'scrape_news_titles' from 'c:\\Users\\cmsup\\Documents\\Python\\news_scraping\\scrape_news_titles.py'>

In [63]:
import requests
from bs4 import BeautifulSoup

url_str = 'https://www.20minutes.fr/'

r = requests.get(url_str, auth=('user', 'pass'))
r.status_code

html_code = r.content.decode('utf')
soup = BeautifulSoup(html_code, 'html.parser')

In [72]:
scrape.view_in_browser(html_code)

In [80]:
tags_with_text = scrape.find_childmost_with_text(soup)

In [96]:
string_list = [
    "Quentin Fillon Maillet",
    "La messe en hommage au militant",
]

class_list = []
class_list = scrape.get_classes_with_strings(tags_with_text, string_list, conflict_index=-1)
new_soup = scrape.highlight_titles(soup, class_list)

In [97]:
scrape.view_in_browser(new_soup)

In [98]:
print(class_list)

['font-weight-bold@xs', 'font-weight-semi-bold@xs']


In [217]:
import scrape_news_titles as scrape
from importlib import reload
reload(scrape_news_titles)

<module 'scrape_news_titles' from 'c:\\Users\\cmsup\\Documents\\Python\\news_scraping\\scrape_news_titles.py'>

In [218]:
list_news_name = [
    'lemonde',
    'lecanard',
    '20min',
]

list_news_url = [
    'https://www.lemonde.fr/',
    'https://www.lecanardenchaine.fr/',
    'https://www.20minutes.fr/',
]

list_news_classes = [
    ['article__title-label', 'article__title'],
    ['article-item__title'],
    ['font-weight-bold@xs', 'font-weight-semi-bold@xs'],
]

In [219]:
all_titles = scrape.scrape_all_news(list_news_name, list_news_url, list_news_classes)

In [ ]:
full_text_list = [''.join(l) for l in all_titles]

In [ ]:
full_text_list

['Le général Oleksandr Syrsky, commandant de l’armée d’Ukraine, au « Monde » : « Les pertes de l’armée russe ont dépassé son niveau de recrutement » Le général Oleksandr Syrsky, commandant de l’armée d’Ukraine, au « Monde » : « Les pertes de l’armée russe ont dépassé son niveau de recrutement »  Donald Trump annonce faire passer ses nouveaux droits de douane mondiaux de 10 % à 15 %La manifestation d’extrême droite en hommage à Quentin Deranque à Lyon se disperse ; une personne interpelléeEmily Harrop et Thibault Anselmet, heureux de l’or olympique et de la « belle image » du ski-alpinisme aux JO 2026Derrière les objectifs de « simplification » pour les agriculteurs, les reculs environnementaux s’accumulentLes chercheurs face aux IA qui « refusent » qu’on les débrancheDes familles face à leur fils à tendance masculiniste : « Il s’est transformé de façon très insidieuse, sans que je le voie »Avec 23 médailles, la délégation française a rempli son objectif aux 

### Look for correlations between sources

In [315]:
import gc
gc.collect(generation=2)

3350

In [346]:
import scrape_news_titles as scrape
from importlib import reload
reload(scrape_news_titles)

<module 'scrape_news_titles' from 'c:\\Users\\cmsup\\Documents\\Python\\news_scraping\\scrape_news_titles.py'>

In [347]:
dict_words, list_freqs = scrape.most_common_words(full_text_list)
common_words, common_freqs = scrape.filter_keywords(dict_words, list_freqs)
processed_words, processed_freq = scrape.preprocess_word_list(common_words, common_freqs)

In [351]:
processed_words

['trump',
 'passer',
 'droits',
 'douane',
 'droite',
 'quentin',
 'deranque',
 'lyon',
 'olympique',
 'leur',
 'tendance',
 'jeux',
 'point',
 'ukraine',
 'remaniement',
 'gouvernement',
 'attendu',
 'sont',
 'chez',
 'entre',
 'rouge',
 'marty',
 'supreme',
 'ligne',
 'images',
 'autour',
 'pouvoir',
 'militaire',
 'depuis',
 'violences',
 'france',
 'semaines',
 'parents',
 'leurs',
 'enfants',
 'toujours',
 'impose',
 'revers',
 'soir',
 'bien',
 'tous',
 'voyage',
 'selon',
 'macron',
 'mort',
 'prend',
 'veut']

In [353]:
word_freq_df = scrape.get_freq_table('Fre')
found_word_freq_df = scrape.get_word_freqs(word_freq_df['BlogFreq'], processed_words).sort_values()